# Free wave packet

*quantum · beginner* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/quantum/free_wave_packet)

A free particle needs no time stepping at all: in momentum space the Schrödinger equation is solved by multiplying each Fourier component by its own phase $e^{-ik^2t/2}$. One FFT, one multiply, one inverse FFT give $\psi$ at any time — for every time at once as a $(T, N)$ tensor. The packet drifts at the group velocity $k_0$ and spreads because its different momenta travel at different speeds.

**Vector form**

$$i\,\partial_t\psi = -\tfrac12\,\partial_x^2\psi\quad\Longrightarrow\quad \tilde\psi(k,t) = e^{-ik^2t/2}\,\tilde\psi(k,0)$$

**Index (tensor) form**

$$\tilde\psi_{k} = F_{kx}\,\psi_x(0),\qquad \tilde\psi_{tk} = K_{tk}\,\tilde\psi_k,\quad K_{tk} = e^{-i k_k^2 t_t/2},\qquad \psi_{tx} = F^{-1}_{xk}\,\tilde\psi_{tk}$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $\psi_x(0)$ | `gaussian_packet(x, x0, sigma0, k0, dx)` | (N,) complex | Gaussian of width $\sigma_0$ moving with momentum $k_0$ |
| $\tilde\psi_k = F_{kx}\psi_x$ | `torch.fft.fft(psi0)` | (N,) complex | repeated $x$ ⇒ summed: change of basis to plane waves $e^{ikx}$ |
| $K_{tk}$ | `exp(-0.5j * einsum('t,k->tk', times, k**2))` | (T, N) complex | free-particle phase of every mode $k$ at every time $t$ (no index summed: an outer product) |
| $\psi_{tx}$ | `torch.fft.ifft(K * psi_k, dim=-1)` | (T, N) complex | back to $x$-space, all $T$ snapshots in one call |
| $\langle x\rangle,\ \sigma$ | `mean_and_width(psi, x, dx)` | (T,), (T,) | centre and spread of $\lvert\psi\rvert^2$ at every time |

Units $\hbar = m = 1$. $\sigma$ is the standard deviation of $\lvert\psi\rvert^2$ (so $\psi \propto \exp(-(x - x_0)^2/(4\sigma_0^2) + ik_0x)$). The exact results are $\langle x\rangle = x_0 + k_0 t$ (group velocity $v_g = dE/dk = k$) and $\sigma(t) = \sigma_0\sqrt{1 + (t/(2\sigma_0^2))^2}$: a narrow packet has a wide spread of momenta and so spreads fast. The dashed lines mark $\langle x\rangle \pm \sigma(t)$ from that formula. The ripples of $\operatorname{Re}\psi$ move at the phase velocity $k/2$, half the speed of the envelope. The grid is periodic, so a packet that runs off one side comes back on the other.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/quantum/free_wave_packet/main.py
"""Main loop: parameters → tensors → solve → plot."""

from pathlib import Path

from problems._common.runtime import cli, device

from . import plot, solve, tensors
from .equation import mean_and_width, norm


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    k0, sigma0 = params["k0"], params["sigma0"]
    x0 = -40.0
    x, k, dx = tensors.grid(4096, 300.0, dev)
    psi0 = tensors.gaussian_packet(x, x0, sigma0, k0, dx)

    times, psi = solve.solve(psi0, k, params["t_max"])
    mean, width = mean_and_width(psi, x, dx)
    mean_exact = solve.exact_centre(times, x0, k0)
    width_exact = solve.exact_width(times, sigma0)

    print(f"<x>(t_max) = {mean[-1]:8.3f}   exact x0 + k0 t = {mean_exact[-1]:8.3f}")
    print(f"sigma(t_max) = {width[-1]:8.3f}   exact sigma0 sqrt(1 + (t/2sigma0^2)^2) = {width_exact[-1]:8.3f}")
    print(f"norm at t_max: {norm(psi[-1], dx):.10f}")
    return plot.animate(x, times, psi, mean_exact, width_exact, out_dir / "free_wave_packet.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/quantum/free_wave_packet/equation.py
r"""Physics → tensor translation.

Free-particle Schrödinger equation (units ħ = m = 1):

    i \partial_t \psi = -\tfrac12 \partial_x^2 \psi.

A plane wave e^{ikx} is an eigenfunction of \partial_x^2 with eigenvalue -k^2, so in the Fourier
(plane-wave) basis the equation decouples into one ODE per mode, i \partial_t \tilde\psi_k = \tfrac12 k^2 \tilde\psi_k,
with the exact solution

    \tilde\psi_{tk} = K_{tk} \tilde\psi_k,      K_{tk} = e^{-i k_k^2 t_t / 2}       (no sum: elementwise)

The changes of basis are the discrete Fourier transform and its inverse (x and k summed):

    \tilde\psi_k = F_{kx} \psi_x(0),            \psi_{tx} = F^{-1}_{xk} \tilde\psi_{tk}

Centre and width of the packet, with \rho_{tx} = |\psi_{tx}|^2 dx:

    \langle x \rangle_t = \rho_{tx} x_x,         \sigma_t^2 = \rho_{tx} (x_x - \langle x \rangle_t)^2     (x summed)
"""

import torch


def kinetic_phase(times: torch.Tensor, k: torch.Tensor) -> torch.Tensor:
    r"""K_{tk} = e^{-i k_k^2 t_t / 2}.   times: (T,), k: (N,) → (T, N) complex"""
    return torch.exp(-0.5j * torch.einsum("t,k->tk", times, k**2))


def evolve(psi0: torch.Tensor, times: torch.Tensor, k: torch.Tensor) -> torch.Tensor:
    r"""\psi_{tx} = F^{-1}_{xk} K_{tk} F_{ky} \psi_y(0).   psi0: (N,) → (T, N) complex"""
    psi_k = torch.fft.fft(psi0)  # (N,)    x → k
    psi_tk = kinetic_phase(times, k) * psi_k  # (T, N)  broadcast over t: each mode gets its own phase
    return torch.fft.ifft(psi_tk, dim=-1)  # (T, N)  k → x for every t at once


def mean_and_width(psi: torch.Tensor, x: torch.Tensor, dx: float) -> tuple[torch.Tensor, torch.Tensor]:
    r"""(\langle x \rangle_t, \sigma_t) of |\psi|^2.   psi: (T, N), x: (N,) → (T,), (T,)"""
    rho = psi.abs() ** 2 * dx  # (T, N) probability per grid cell
    mean = torch.einsum("tx,x->t", rho, x)  # (T,)
    spread = x[None, :] - mean[:, None]  # (T, N)
    width = torch.sqrt(torch.einsum("tx,tx->t", rho, spread**2))  # (T,)
    return mean, width


def norm(psi: torch.Tensor, dx: float) -> torch.Tensor:
    r"""\int |\psi|^2 dx \approx \psi^*_x \psi_x dx  (x summed).   (..., N) → (...)"""
    return torch.einsum("...x,...x->...", psi.conj(), psi).real * dx


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/quantum/free_wave_packet/tensors.py
"""Build the concrete tensors: periodic grid, its wavenumbers and the initial Gaussian wave packet."""

import math

import torch


def grid(n: int, length: float, device: torch.device) -> tuple[torch.Tensor, torch.Tensor, float]:
    """Periodic grid x ∈ [−length/2, length/2) and its FFT wavenumbers k. Returns (x (N,), k (N,), dx)."""
    dx = length / n
    x = -length / 2 + dx * torch.arange(n, dtype=torch.float64, device=device)  # (N,)
    k = 2 * math.pi * torch.fft.fftfreq(n, d=dx, dtype=torch.float64, device=device)  # (N,) in FFT order
    return x, k, dx


def gaussian_packet(x: torch.Tensor, x0: float, sigma0: float, k0: float, dx: float) -> torch.Tensor:
    """ψ_x = exp(−(x−x0)²/(4σ0²) + i k0 x), normalised to Σ|ψ|² dx = 1. σ0 is the width of |ψ|². Shape (N,) complex."""
    psi = torch.exp(-((x - x0) ** 2) / (4 * sigma0**2) + 1j * k0 * x)
    return psi / torch.sqrt((psi.abs() ** 2).sum() * dx)


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/quantum/free_wave_packet/solve.py
"""Evaluate the exact free evolution at every saved time, and the textbook centre and width."""

import torch

from .equation import evolve


def solve(psi0: torch.Tensor, k: torch.Tensor, t_max: float, n_steps: int = 240) -> tuple[torch.Tensor, torch.Tensor]:
    """Return (times (T,), ψ (T, N) complex). No time stepping: the k-space solution is exact at any t."""
    times = torch.linspace(0.0, t_max, n_steps + 1, dtype=torch.float64, device=psi0.device)  # (T,)
    return times, evolve(psi0, times, k)


def exact_centre(times: torch.Tensor, x0: float, k0: float) -> torch.Tensor:
    """⟨x⟩(t) = x0 + k0 t: the packet moves at the group velocity v_g = k0 (ħ = m = 1). Shape (T,)."""
    return x0 + k0 * times


def exact_width(times: torch.Tensor, sigma0: float) -> torch.Tensor:
    """σ(t) = σ0 √(1 + (t / (2σ0²))²) for the standard deviation of |ψ|². Shape (T,)."""
    return sigma0 * torch.sqrt(1 + (times / (2 * sigma0**2)) ** 2)


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/quantum/free_wave_packet/plot.py
"""Animate |ψ|² and Re ψ of the drifting, spreading packet, with the predicted ⟨x⟩ ± σ(t) marked."""

from pathlib import Path

import matplotlib.pyplot as plt
import torch

from problems._common.plotting import ACCENT, frame_indices, save_gif


def animate(
    x: torch.Tensor,
    times: torch.Tensor,
    psi: torch.Tensor,
    mean: torch.Tensor,
    width: torch.Tensor,
    path: Path,
) -> Path:
    xs = x.cpu().numpy()
    psi = psi.cpu()
    prob = (psi.abs() ** 2).numpy()  # (T, N)
    re = psi.real.numpy()  # (T, N)
    t = times.cpu().numpy()
    xm, sig = mean.cpu().numpy(), width.cpu().numpy()  # (T,), (T,)
    lo = min(xm.min() - 3 * sig.max(), xm[0] - 10)
    hi = max(xm.max() + 3 * sig.max(), xm[0] + 10)

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.set_xlim(max(lo, xs[0]), min(hi, xs[-1]))
    amp = float(abs(re).max())
    ax.set_ylim(-1.1 * amp, 1.25 * amp)
    ax.set_xlabel("x")
    ax.set_ylabel("Re ψ,  |ψ|²")

    (line_re,) = ax.plot([], [], color=ACCENT[0], lw=0.9, alpha=0.8, label="Re ψ")
    (line_p,) = ax.plot([], [], color="#111827", lw=2.2, label="|ψ|²")
    fill = [ax.fill_between(xs, 0 * xs, color="#111827", alpha=0.1)]
    centre = ax.axvline(0, color=ACCENT[1], lw=1.2, label="x0 + k0 t")
    edges = [ax.axvline(0, color=ACCENT[1], lw=1, ls="--") for _ in range(2)]
    edges[0].set_label("± σ(t)")
    clock = ax.text(0.02, 0.95, "", transform=ax.transAxes, va="top", family="monospace")
    ax.legend(loc="upper right", fontsize=8)

    def update(step: int):
        line_re.set_data(xs, re[step])
        line_p.set_data(xs, prob[step])
        fill[0].remove()
        fill[0] = ax.fill_between(xs, prob[step], color="#111827", alpha=0.1)
        centre.set_xdata([xm[step], xm[step]])
        edges[0].set_xdata([xm[step] - sig[step]] * 2)
        edges[1].set_xdata([xm[step] + sig[step]] * 2)
        clock.set_text(f"t = {t[step]:5.1f}\nσ = {sig[step]:5.2f}")

    fig.tight_layout()
    return save_gif(fig, update, frame_indices(len(t)), path)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'k0': 2.0,
    'sigma0': 1.5,
    't_max': 30.0
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.quantum.free_wave_packet")]:
    del sys.modules[name]
main = importlib.import_module("problems.quantum.free_wave_packet.main")
problem_dir = Path("problems/quantum/free_wave_packet")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
